In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [2]:
train = pd.read_csv("./data/mpg_train.csv")
test = pd.read_csv("./data/mpg_test.csv") # no mpg column -> we must predict it
train

,id,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,1,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,3,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
2,4,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
3,6,15.0,8,429.0,198.0,4341,10.0,70,usa,ford galaxie 500
4,7,14.0,8,454.0,220.0,4354,9.0,70,usa,chevrolet impala
...,...,...,...,...,...,...,...,...,...,...
313,394,27.0,4,140.0,86.0,2790,15.6,82,usa,ford mustang gl
314,395,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
315,396,32.0,4,135.0,84.0,2295,11.6,82,usa,dodge rampage
316,397,28.0,4,120.0,79.0,2625,18.6,82,usa,ford ranger


In [3]:
train.describe()

,id,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
count,318.000000,318.000000,318.000000,318.000000,315.000000,318.000000,318.000000,318.000000
mean,197.899371,23.449057,5.462264,194.309748,104.857143,2973.729560,15.619497,75.949686
std,114.885086,7.890783,1.719298,106.979003,40.065018,871.718895,2.814350,3.699772
min,1.000000,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000
25%,101.500000,17.000000,4.000000,98.000000,75.000000,2212.250000,13.900000,73.000000
50%,198.500000,23.000000,4.000000,144.500000,92.000000,2764.500000,15.500000,76.000000
75%,298.250000,29.000000,8.000000,302.000000,129.500000,3641.750000,17.300000,79.000000
max,398.000000,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000


In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            318 non-null    int64  
 1   mpg           318 non-null    float64
 2   cylinders     318 non-null    int64  
 3   displacement  318 non-null    float64
 4   horsepower    315 non-null    float64
 5   weight        318 non-null    int64  
 6   acceleration  318 non-null    float64
 7   model_year    318 non-null    int64  
 8   origin        318 non-null    str    
 9   name          318 non-null    str    
dtypes: float64(4), int64(4), str(2)
memory usage: 25.0 KB


In [5]:
train.isna().sum()

id              0
mpg             0
cylinders       0
displacement    0
horsepower      3
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64

In [6]:
test.isna().sum() # horsepower missing here too -> must fill, can't drop

id              0
cylinders       0
displacement    0
horsepower      3
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64

In [7]:
sorted(train["name"].str.split().str[0].unique()) # first word = brand, look for typos

['amc',
 'audi',
 'bmw',
 'buick',
 'cadillac',
 'capri',
 'chevrolet',
 'chevy',
 'chrysler',
 'datsun',
 'dodge',
 'fiat',
 'ford',
 'hi',
 'honda',
 'maxda',
 'mazda',
 'mercedes-benz',
 'mercury',
 'nissan',
 'oldsmobile',
 'opel',
 'peugeot',
 'plymouth',
 'pontiac',
 'renault',
 'saab',
 'subaru',
 'toyota',
 'toyouta',
 'triumph',
 'vokswagen',
 'volkswagen',
 'volvo',
 'vw']

In [8]:
hp_median = train["horsepower"].median() # learn from train only

train["horsepower"] = train["horsepower"].fillna(hp_median)
test["horsepower"] = test["horsepower"].fillna(hp_median) # same value for test

In [9]:
fix_brand = {"chevroelt": "chevrolet", "chevy": "chevrolet", "vw": "volkswagen", "vokswagen": "volkswagen",
	"toyouta": "toyota", "maxda": "mazda", "mercedes-benz": "mercedes"}

train["brand"] = train["name"].str.split().str[0].replace(fix_brand)
test["brand"] = test["name"].str.split().str[0].replace(fix_brand)

In [10]:
le_origin, le_brand = LabelEncoder(), LabelEncoder()

train["origin"] = le_origin.fit_transform(train["origin"]) # europe, japan, usa -> 0, 1, 2
test["origin"] = le_origin.transform(test["origin"]) # transform only, same mapping

train["brand"] = le_brand.fit_transform(train["brand"])
test["brand"] = le_brand.transform(test["brand"])

In [11]:
X = train.drop(columns=["id", "name", "mpg"])
y = train["mpg"]
X_sub = test.drop(columns=["id", "name"]) # features of the data to submit

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
	X, y, test_size=0.2, random_state=42)

In [13]:
X_train

,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,brand
209,6,232.0,90.0,3210,17.2,78,2,0
76,8,455.0,225.0,4951,11.0,73,2,3
94,4,116.0,75.0,2158,15.5,73,0,19
221,6,163.0,125.0,3140,13.6,78,0,29
310,4,156.0,92.0,2585,14.5,82,2,7
...,...,...,...,...,...,...,...,...
188,8,400.0,190.0,4325,12.2,77,2,7
71,8,302.0,137.0,4042,14.5,73,2,11
106,4,71.0,65.0,1836,21.0,74,1,26
270,3,70.0,100.0,2420,12.5,80,1,14


In [14]:
scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train) # fit on TRAIN only
Xval = scaler.transform(X_val)
Xsub = scaler.transform(X_sub)

In [15]:
model = MLPRegressor(hidden_layer_sizes=(20,), max_iter=3000, random_state=42)
model.fit(Xtr, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(20,)"
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",3000
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'poisson'}, default='squared_error'The loss function to use when training the weights. Note that the""squared error"" and ""poisson"" losses actually implement""half squares error"" and ""half poisson deviance"" to simplify thecomputation of the gradient. Furthermore, the ""poisson"" loss internally usesa log-link (exponential as the output activation function) and requires``y >= 0``... versionchanged:: 1.7 Added parameter `loss` and option 'poisson'.",'squared_error'
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the regressor will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate ``learning_rate_`` at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when solver='sgd'.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5


In [16]:
y_pred_train = model.predict(Xtr)
y_pred_val = model.predict(Xval)

print(f"Train  RMSE: {root_mean_squared_error(y_train, y_pred_train):.2f}  R2: {r2_score(y_train, y_pred_train):.3f}")
print(f"Val    RMSE: {root_mean_squared_error(y_val, y_pred_val):.2f}  R2: {r2_score(y_val, y_pred_val):.3f}")

Train  RMSE: 2.53  R2: 0.899
Val    RMSE: 1.94  R2: 0.934


In [17]:
SAVE_CSV = False

submission_df = test[["id"]].copy()
submission_df["mpg"] = model.predict(Xsub)
if SAVE_CSV:
	submission_df.to_csv("submission.csv", index=False)

submission_df

,id,mpg
0,2,14.770268
1,5,15.858942
2,13,15.212776
3,16,20.928386
4,18,20.025390
...,...,...
75,379,36.849258
76,380,34.442632
77,387,25.136178
78,390,26.565881
